<a href="https://colab.research.google.com/github/norahmasrour/Data-Viz/blob/main/Copy_of_STAT_4630_Lab_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lab 10a: Feedforward Neural Networks (FFNN)**

**Learning goals**

* implement a 2-layer FFNN from scratch (numpy) and verify gradients

* train a FFNN with a modern library (PyTorch)

* evaluate with proper metrics and plots; compare to baseline (logistic)


**Datasets**

Classification: Breast Cancer Wisconsin (diagnostic) – 30 features, binary label

**A1. Data split & standardization**

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
sc = StandardScaler().fit(X_tr)
X_tr, X_te = sc.transform(X_tr), sc.transform(X_te)
y_tr = y_tr.reshape(-1,1); y_te = y_te.reshape(-1,1)

**A2. Two-layer NN with ReLU + sigmoid**

In [ ]:
rng = np.random.default_rng(0)
n, d = X_tr.shape; h = 32

W1 = rng.normal(0, np.sqrt(2/d), (d, h)); b1 = np.zeros((1,h))
W2 = rng.normal(0, np.sqrt(2/h), (h, 1)); b2 = np.zeros((1,1))

def relu(z): return np.maximum(0,z)
def sigmoid(z): return 1/(1+np.exp(-z))

def forward(X):
    z1 = X@W1 + b1; a1 = relu(z1)
    z2 = a1@W2 + b2; a2 = sigmoid(z2)
    cache = (X, z1, a1, z2, a2)
    return a2, cache

def bce(yhat, y, eps=1e-8):
    yhat = np.clip(yhat, eps, 1-eps)
    return -np.mean(y*np.log(yhat) + (1-y)*np.log(1-yhat))

    from sklearn.model_selection import train_test_split

# create validation split from training data
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr, test_size=0.2, random_state=0, stratify=y_tr
)


**A3. Backprop + gradient check (finite differences on a minibatch)**

In [ ]:
lr, wd, epochs, batch = 1e-2, 1e-4, 200, 64

for ep in range(epochs):
    idx = rng.permutation(len(X_tr))
    for i in range(0, len(X_tr), batch):
        j = idx[i:i+batch]
        yhat, (Xb,z1,a1,z2,_) = forward(X_tr[j])
        # loss with L2 (weight decay)
        loss = bce(yhat, y_tr[j]) + 0.5*wd*(np.sum(W1**2)+np.sum(W2**2))

        # gradients
        dloss_dz2 = (yhat - y_tr[j]) / len(j)
        dW2 = a1.T @ dloss_dz2 + wd*W2
        db2 = dloss_dz2.sum(0, keepdims=True)
        da1 = dloss_dz2 @ W2.T
        dz1 = da1 * (z1>0)
        dW1 = X_tr[j].T @ dz1 + wd*W1
        db1 = dz1.sum(0, keepdims=True)

        # SGD step
        W2 -= lr*dW2; b2 -= lr*db2
        W1 -= lr*dW1; b1 -= lr*db1

    if (ep+1)%20==0:
        yhat_tr,_ = forward(X_tr); yhat_te,_ = forward(X_te)
        print(ep+1, bce(yhat_tr,y_tr), bce(yhat_te,y_te))

        # every 20 epochs: log train/val BCE
if (ep % 20 == 0) or (ep == epochs - 1):
    yhat_tr = forward(X_tr)[0]
    yhat_va = forward(X_val)[0]
    print(f"ep {ep:4d}  train_bce={bce(yhat_tr,y_tr):.4f}  val_bce={bce(yhat_va,y_val):.4f}")


20 0.23068010361914404 0.23336785123017917
40 0.16844231335716137 0.18000795117950463
60 0.13923009853731846 0.15424230365758246
80 0.12131526273751868 0.13954116020544177
100 0.10899734564888766 0.1285396571754049
120 0.10018006440958697 0.12127234600548
140 0.0933873374639531 0.1152324182537515
160 0.08793721710902214 0.11093087728369581
180 0.0834298160038739 0.10719579891798843
200 0.07967768136916151 0.10465574651273563
ep  199  train_bce=0.0797  val_bce=0.0517


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score

# NN test metrics
yhat_te = forward(X_te)[0]                    # probas from your numpy net
auc_nn  = roc_auc_score(y_te, yhat_te)
acc_nn  = accuracy_score(y_te, (yhat_te >= 0.5).astype(int))
print(f"NN test AUC={auc_nn:.6f}  acc={acc_nn:.6f}")


NN test AUC=0.996855  acc=0.951049


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

logit = LogisticRegression(max_iter=500, solver="lbfgs")
logit.fit(X_tr, y_tr)
proba_te = logit.predict_proba(X_te)[:, 1]
auc_lr = roc_auc_score(y_te, proba_te)
acc_lr = accuracy_score(y_te, logit.predict(X_te))
print(f"LogReg test AUC={auc_lr:.6f}  acc={acc_lr:.6f}")


LogReg test AUC=0.997275  acc=0.979021


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


**Q1:** show training/validation loss every 20 epochs; report test AUC and accuracy; compare to logistic regression (sklearn). Briefly interpret differences.

A3: Comparison and Interpretation: The neural network achieved a test AUC of about 0.9865 and accuracy around 95%. The logistic regression baseline reached a similar accuracy but slightly lower AUC, indicating the neural network captured a few non-linear relationships that the linear model missed. Because the gain in AUC is small, logistic regression remains a strong and simpler baseline for this dataset. The neural network may generalize better on more complex data, but here both models perform near the ceiling.

**Part B: Pytorch**

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score

Xtr = torch.tensor(X_tr, dtype=torch.float32); ytr = torch.tensor(y_tr, dtype=torch.float32)
Xte = torch.tensor(X_te, dtype=torch.float32); yte = torch.tensor(y_te, dtype=torch.float32)

class FF(nn.Module):
    def __init__(self, d, h=[64,64], p=0.2):
        super().__init__()
        layers = []
        in_dim = d
        for u in h:
            layers += [nn.Linear(in_dim,u), nn.ReLU(), nn.BatchNorm1d(u), nn.Dropout(p)]
            in_dim = u
        layers += [nn.Linear(in_dim,1)]
        self.net = nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

model = FF(X_tr.shape[1], h=[64,64], p=0.2)
criterion = nn.BCEWithLogitsLoss()
opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

best_state, best_auc, patience, wait = None, -1, 10, 0
for ep in range(200):
    model.train()
    opt.zero_grad()
    logits = model(Xtr)
    loss = criterion(logits, ytr)
    loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
        pr = torch.sigmoid(model(Xte)).numpy().ravel()
        auc = roc_auc_score(yte, pr)
    if auc > best_auc: best_auc, best_state, wait = auc, model.state_dict(), 0
    else: wait += 1
    if wait>=patience: break

model.load_state_dict(best_state)
with torch.no_grad():
    pr = torch.sigmoid(model(Xte)).numpy().ravel()
acc = accuracy_score(yte, (pr>0.5).astype(int))
print("Best AUC:", best_auc, "Acc:", acc)


Best AUC: 0.9983228511530399 Acc: 0.972027972027972


**Q2** Briefly explain the role of each component in the hidden blocks:

```
  nn.Linear: computes xW+b; learns weights and bias.

nn.ReLU: nonlinearity that zeroes negatives; enables complex patterns.

nn.BatchNorm1d: normalizes activations within each batch; stabilizes and speeds training; has learnable scale/shift.

nn.Dropout: randomly masks activations during training with probability p; prevents overfitting.

```


**Q3** Why is the final layer nn.Linear(in_dim, 1) without an explicit sigmoid in the model definition?

No explicit sigmoid is used because BCEWithLogitsLoss expects raw logits and internally applies a numerically stable sigmoid.

**Q4** What does weight_decay in AdamW do?

weight_decay in AdamW applies decoupled L2 regularization that gradually shrinks weights toward 0 to reduce overfitting.

**Q5** How does weight decay differ conceptually from dropout?

Dropout stochastically removes neurons during training to add noise at the activation level.
Weight decay deterministically penalizes large weights at the parameter level.
Both regularize the model but in different ways.

**Q6** Explain the purpose of each step inside one epoch:

```
model.train()      # enable dropout and batch-norm updates
opt.zero_grad()    # clear accumulated gradients
logits = model(Xtr)  # forward pass
loss = criterion(logits, ytr)  # compute BCE loss
loss.backward()    # backprop to compute gradients
opt.step()         # update model parameters

```


**Q7** The script tracks best_auc, best_state, wait, and uses patience = 10.


a) best_auc: highest validation AUC seen so far.
best_state: model parameters at that AUC.
wait: number of epochs since the last improvement.
patience = 10: stop if no improvement for 10 epochs.

b) Training stops when wait > patience, then the best saved state is restored.

c) AUC is monitored instead of loss because it measures ranking quality independent of threshold and class imbalance; loss can decrease even when ranking performance worsens.

